In [1]:
import pickle
import pandas as pd 

import numpy as np 
import os 


import pickle


import scanpy as sc

import pandas as pd



import matplotlib.pyplot as plt
from matplotlib import colors
# color_map
# sc.settings.set_figure_params(dpi=120)

plt.rcParams['figure.figsize']=(8,8) #rescale figures
# sc.settings.verbosity = 3
import IPython.display
from matplotlib_inline.backend_inline import set_matplotlib_formats
IPython.display.set_matplotlib_formats = set_matplotlib_formats

sc.set_figure_params(scanpy=True, dpi_save=400,dpi=150)

plt.rcParams["font.family"] = "Arial"
plt.rcParams['pdf.fonttype'] = 42





In [2]:
sc.settings.figdir = '../fig4a_main_realslices'

figdir =  '../fig4a_main_realslices'

## plot DE genes for Inv, DCIS, Myoep

In [3]:
dataset_full = 'dataset12_xenium'


scrna_path = '../../datasets/' + dataset_full + '/data/'

adata_scrna = sc.read(scrna_path + "scrna_ref_norm1knolog.h5ad")

In [4]:
ct_reanno = {'Invasive_Tumor': 'InvTumor',
             'Macrophages_1':'Macrophages',
             'Macrophages_2':'Macrophages',
             'CD4+_T_Cells':'CD4+_T_Cell',
             'Stromal':'Stromal',
             'DCIS_2':'DCIS',
             'DCIS_1':'DCIS',
             'CD8+_T_Cells':'CD8+_T_Cell',
             'B_Cells':'BCells',
             'Prolif_Invasive_Tumor': 'InvTumor',
             'Myoepi_ACTA2+':'Myoepi_ACTA2+',
                'Myoepi_KRT15+':'Myoepi_KRT15+',
                'Endothelial':'Endothelial',
             'Perivascular-Like':'Perivascular-Like',
               'IRF7+_DCs':'DCs',
               'LAMP3+_DCs':'DCs',
               'Mast_Cells':'Mast'
             }

scrna_reanno = []
for entry in adata_scrna.obs['celltype'].values:
    scrna_reanno.append(ct_reanno[entry])
adata_scrna.obs['Anno_l2'] = scrna_reanno

In [5]:
adata_scrna_de = adata_scrna.copy()
sc.pp.log1p(adata_scrna_de)

In [6]:
# sc.pl.tsne(adata_scrna_de, color='Anno_l2')


sc.tl.rank_genes_groups(adata_scrna_de, groupby='Anno_l2',groups=['DCIS','InvTumor','Myoepi_ACTA2+', 'Myoepi_KRT15+'], method="wilcoxon")
sc.tl.filter_rank_genes_groups(adata_scrna_de,   key_added='rank_genes_groups_filtered',  min_fold_change=1.0)
all_genes = sc.get.rank_genes_groups_df(adata_scrna_de,group=None, key='rank_genes_groups_filtered', pval_cutoff=0.05, log2fc_min=1.0)
all_genes = all_genes[~all_genes['names'].isna()]


In [6]:
adata_concat = sc.read(figdir + '/adata_concat2_tmp_save.h5ad')

adata_spatial_log = adata_concat.copy()

sc.pp.log1p(adata_spatial_log)

/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
to_plot = list(set(all_genes['names']))

In [12]:
len(to_plot)

101

In [13]:

obs = adata_scrna_de[:,to_plot].X.toarray()
obs = pd.DataFrame(obs,columns=to_plot,index=adata_scrna_de.obs['Anno_l2'])
average_obs = obs.groupby(level=0).mean()

/tmp/ipykernel_3145504/3726494875.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  average_obs = obs.groupby(level=0).mean()


In [14]:

to_plot_dict = {}
for entry in list(set(all_genes['group'])):
    to_plot_dict[entry] = []

for entry in to_plot:
    tmp_df = all_genes[all_genes['names']==entry]
    tmp_groups = list(all_genes[all_genes['names']==entry]['group'].values)
    if len(tmp_groups) == 1: # only 1 gene
        to_plot_dict[tmp_groups[0]].append(entry)
    else:
        # assign to group with max expression 
        to_plot_dict[average_obs.loc[list(all_genes[all_genes['names']==entry]['group'].values), entry].idxmax()].append(entry)


# sort genes for the same group by their max expression 
for entry in to_plot_dict.keys():
    to_plot_dict[entry] = list(average_obs.loc[entry,to_plot_dict[entry]].sort_values(ascending=False).index)




In [8]:
tmp_tumor = list(adata_spatial_log.obs[['deconv_prop_Tumor_region_classify_-1',
       'deconv_prop_Tumor_region_classify_DCIS',
       'deconv_prop_Tumor_region_classify_InvTumor']].astype(int).idxmax(axis=1).values)

tmp_tumor = [entry.split('_')[-1] for entry in tmp_tumor]


adata_spatial_log.obs['Tumor_Class_DE'] = tmp_tumor



In [10]:
adata_spatial_log.obs['Myoep_Class_DE'] = 0.5


In [11]:

tmp_myo = list(adata_spatial_log.obs[['deconv_prop_Myoepithelial_region_classify_Myo_KRT15+',
       'deconv_prop_Myoepithelial_region_classify_Myo_ACTA2+','Myoep_Class_DE']].idxmax(axis=1).values)

tmp_myo = [entry.split('_')[-1] for entry in tmp_myo]


adata_spatial_log.obs['Myoep_Class_DE'] = tmp_myo




In [12]:
adata_concat2 = adata_concat.copy()

/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [13]:
adata_spatial_log[adata_spatial_log.obs['Myoep_Class_DE']=='DE'].obs['Myoep_Class_DE'] = '-1'

/tmp/ipykernel_2040877/3446484050.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_spatial_log[adata_spatial_log.obs['Myoep_Class_DE']=='DE'].obs['Myoep_Class_DE'] = '-1'
/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [14]:
adata_concat2.obs = adata_spatial_log.obs.copy()

In [15]:
adata_concat2[adata_concat2.obs['Myoep_Class_DE']=='DE'].obs['Myoep_Class_DE'] = '-1'

/tmp/ipykernel_2040877/1281377411.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_concat2[adata_concat2.obs['Myoep_Class_DE']=='DE'].obs['Myoep_Class_DE'] = '-1'
/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [16]:
adata_spatial_log.obs['Tumor_myoep_for_DE'] = adata_spatial_log.obs['Tumor_Class_DE'].astype(str) + '_' + adata_spatial_log.obs['Myoep_Class_DE'].astype(str)

In [17]:
adata_concat2.obs['Tumor_myoep_for_DE'] = adata_concat2.obs['Tumor_Class_DE'].astype(str) + '_' + adata_concat2.obs['Myoep_Class_DE'].astype(str)

In [19]:
adata_spatial_log2 = adata_spatial_log[adata_spatial_log.obs['Tumor_myoep_for_DE']!='-1_DE']




In [20]:
adata_concat2 = adata_concat2[adata_concat2.obs['Tumor_myoep_for_DE']!='-1_DE']




In [21]:
# del adata_spatial_log2.uns['dendrogram_Tumor_myoep_for_DE']

In [22]:
adata_spatial_log2.obs['Tumor_myoep_for_DE'] = adata_spatial_log2.obs['Tumor_myoep_for_DE'].astype('category')

/tmp/ipykernel_2040877/2166530007.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_spatial_log2.obs['Tumor_myoep_for_DE'] = adata_spatial_log2.obs['Tumor_myoep_for_DE'].astype('category')
/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [23]:
adata_concat2.obs['Tumor_myoep_for_DE'] = adata_concat2.obs['Tumor_myoep_for_DE'].astype('category')

/tmp/ipykernel_2040877/3072577992.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_concat2.obs['Tumor_myoep_for_DE'] = adata_concat2.obs['Tumor_myoep_for_DE'].astype('category')
/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/project/mlobo6/miniconda3/envs/spadecoder/lib/python3.12/site-packages/anndata/_core/anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [24]:
categories_order = [ 'DCIS_DE',  'DCIS_KRT15+', 'DCIS_ACTA2+',  '-1_ACTA2+','InvTumor_ACTA2+',  
        'InvTumor_DE', 'InvTumor_KRT15+', '-1_KRT15+',]

In [ ]:
sc.pl.matrixplot(adata_concat2, to_plot_dict, categories_order=categories_order, var_group_rotation=0, save='_final_subfig_in_4.pdf',groupby="Tumor_myoep_for_DE",  standard_scale='var')